# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Track 4 · Ngày 1** — Forest CoverType, MLP `M-base` (54→256→128→7, 47 879 tham số).


## 0. Cấu hình môi trường và đường dẫn

Đặt seed cố định, chọn device (CPU ở đây vì máy không có GPU)

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_2A202602859/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device =", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("threads:", torch.get_num_threads())

# Thêm code/ vào sys.path để import các module trong cùng thư mục
sys.path.insert(0, os.path.abspath("."))

# --- W&B bật khi có WANDB_API_KEY trong môi trường, để notebook không bao giờ treo chờ đăng nhập khi chấm bài. Không có key -> bỏ qua, vẫn ghi PNG + JSON.
USE_WANDB = (os.environ.get("WANDB_MODE", "") != "disabled") and bool(os.environ.get("WANDB_API_KEY"))
WANDB_PROJECT = "K4-Track4-Day1-Neuralnetwork"
if USE_WANDB:
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=False)
        print("wandb sẵn sàng:", wandb.__version__)
    except Exception as e:
        USE_WANDB = False
        print("wandb không dùng được ->", type(e).__name__, str(e)[:120])
else:
    print("W&B tắt (không có WANDB_API_KEY) — sản phẩm nộp vẫn là figures/*.png + results/*.json.")


torch 2.14.0+cpu | device = cpu
threads: 6
W&B tắt (không có WANDB_API_KEY) — sản phẩm nộp vẫn là figures/*.png + results/*.json.


In [ ]:
from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import (DEFAULT_CFG, set_seed, evaluate, predict, run_experiment,
                   compute_loss, macro_f1_from_confusion, final_eval)
from plots import plot_run, plot_compare, plot_confusion, plot_loss_curve
from results_table import save_result, load_results, to_row, write_xlsx
from experiments import EXPERIMENTS, BASE

def get_or_run(cfg, data, force=False):
    """Nếu results/<exp_id>.json đã có thì nạp lại; ngược lại huấn luyện rồi lưu JSON + ảnh + log W&B."""
    exp_id = cfg["exp_id"]
    jpath = f"{OUT_DIR}/results/{exp_id}.json"
    fpath = f"{OUT_DIR}/figures/{exp_id}.png"
    if (not force) and os.path.exists(jpath) and os.path.exists(fpath):
        rec = json.load(open(jpath, encoding="utf-8"))
        rec["_cached"] = True
        return rec
    result = run_experiment(cfg, data)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, fpath)
    if USE_WANDB:
        try:
            run = wandb.init(project=WANDB_PROJECT, name=exp_id, group=cfg.get("group"),
                             reinit=True, config={k: str(v) for k, v in cfg.items()})
            for i, e in enumerate(result["history"]["epoch"]):
                run.log({k: result["history"][k][i] for k in
                         ("train_loss","val_loss","val_acc","val_macro_f1","grad_norm",
                          "grad_norm_max","clip_frac","epoch_time_s")}, step=e)
            run.log({"curves": wandb.Image(fpath)})
            run.summary.update({k: result["summary"][k] for k in result["summary"]})
            wandb.finish()
        except Exception as e:
            print("  [wandb] lỗi:", type(e).__name__, str(e)[:120])
    return result


## Part 0 — Dữ liệu

Chia sẵn bằng `data/split_metadata.csv`: `train` 464 809 / `eval` 116 203.
Tách validation 20% từ train (phân tầng, seed 42) → 371 847 train / 92 962 val.
Chuẩn hoá chỉ 10 cột số, bằng mean/std của phần train còn lại (không dùng val/eval → tránh rò rỉ).


In [ ]:
if not os.path.exists(f"{REPO_ROOT}/data/processed/train.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

Xtr = data["X_tr"][:, :10]
print("\nmean(10 cột số, train) ≈", np.round(Xtr.mean(0).cpu().numpy(), 3)[:5], "...")
print("std (10 cột số, train) ≈", np.round(Xtr.std(0).cpu().numpy(), 3)[:5], "...")
# Tỉ lệ lớp ở 3 tập gần như bằng nhau
for name in ("y_tr", "y_val", "y_eval"):
    b = np.bincount(data[name].cpu().numpy(), minlength=7) / len(data[name])
    print(f"tỉ lệ lớp {name:6s}:", np.round(b, 4))


train full : (464809, 54)
  -> train : (371847, 54)
  -> val   : (92962, 54)
eval       : (116203, 54)
lớp đa số  : 1; accuracy 'luôn đoán lớp đa số' trên val = 0.4876 (mốc thấp nhất phải vượt)

mean(10 cột số, train) ≈ [-0. -0.  0.  0.  0.] ...
std (10 cột số, train) ≈ [1. 1. 1. 1. 1.] ...
tỉ lệ lớp y_tr  : [0.3646 0.4876 0.0615 0.0047 0.0163 0.0299 0.0353]
tỉ lệ lớp y_val : [0.3646 0.4876 0.0615 0.0047 0.0163 0.0299 0.0353]
tỉ lệ lớp y_eval: [0.3646 0.4876 0.0615 0.0047 0.0163 0.0299 0.0353]


## Part 1 — Model 



In [ ]:
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n = count_params(model)
print("M-base params =", n, "| kỳ vọng", EXPECTED_PARAMS[(256, 128)])
assert n == EXPECTED_PARAMS[(256, 128)]
for h in [(512, 256), (256, 128, 64)]:
    m = MLP(hidden=h, init="he")
    print(f"hidden {h}: params={count_params(m)} kỳ vọng {EXPECTED_PARAMS[h]}")
    assert count_params(m) == EXPECTED_PARAMS[h]

x8 = torch.randn(8, 54, device=device)
logits = model(x8)
print("logits shape:", tuple(logits.shape)); assert tuple(logits.shape) == (8, 7)

m0 = evaluate(model, data["X_val"], data["y_val"], "ce")
print(f"\n[1] loss bước 0 = {m0['loss']:.4f}  (ln 7 = {np.log(7):.4f})")

# gradient 
model.train()
loss = compute_loss(model(x8), torch.randint(0, 7, (8,), device=device), "ce")
loss.backward()
print("\n[3] grad norm từng tham số:")
for name, p in model.named_parameters():
    g = p.grad
    assert g is not None and torch.isfinite(g).all() and g.norm().item() > 0, name
    print(f"    {name:16s} {g.norm().item():.4e}")
print("-> mọi tham số có gradient khác 0.")


M-base params = 47879 | kỳ vọng 47879
hidden (512, 256): params=161287 kỳ vọng 161287
hidden (256, 128, 64): params=55687 kỳ vọng 55687
logits shape: (8, 7)

[1] loss bước 0 = 2.2691  (ln 7 = 1.9459)

[3] grad norm từng tham số:
    net.0.weight     2.6666e+00
    net.0.bias       3.5557e-01
    net.2.weight     6.1083e+00
    net.2.bias       4.0876e-01
    net.4.weight     3.6639e+00
    net.4.bias       3.8007e-01
-> mọi tham số có gradient khác 0.


In [ ]:
set_seed(0)
small = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt = torch.optim.SGD(small.parameters(), lr=0.5)
X20 = data["X_tr"][:20]; y20 = data["y_tr"][:20]
losses = []
for step in range(400):
    opt.zero_grad(set_to_none=True)
    loss = compute_loss(small(X20), y20, "ce")
    loss.backward(); opt.step()
    losses.append(float(loss.item()))
small.eval()
with torch.no_grad():
    acc20 = (small(X20).argmax(1) == y20).float().mean().item()
print(f"[2] loss 20 mẫu: {losses[0]:.4f} -> {losses[-1]:.6f} | accuracy = {acc20:.3f}")
plot_loss_curve(losses, f"{OUT_DIR}/figures/part1_overfit20.png",
                title="Quá khớp 20 mẫu (M-base, SGD lr=0.5, 400 bước)", xlabel="bước")
print("đã lưu figures/part1_overfit20.png")

print("\nĐộ lệch chuẩn kích hoạt sau mỗi ReLU (bước 0, lô val 4096):")
for init in ("he", "xavier", "normal", "zeros", "default"):
    mm = MLP(hidden=(256, 128), init=init).to(device)
    st = activation_stats(mm, data["X_val"][:4096])
    l0 = evaluate(mm, data["X_val"][:20000], data["y_val"][:20000], "ce")["loss"]
    print(f"  {init:8s} std={[round(v,4) for v in st]}  loss_bước0={l0:.4f}")


[2] loss 20 mẫu: 1.9668 -> 0.000449 | accuracy = 1.000


đã lưu figures/part1_overfit20.png

Độ lệch chuẩn kích hoạt sau mỗi ReLU (bước 0, lô val 4096):
  he       std=[0.4049, 0.3901]  loss_bước0=2.2560
  xavier   std=[0.1665, 0.1371]  loss_bước0=1.8670
  normal   std=[0.0214, 0.0023]  loss_bước0=1.9459
  zeros    std=[0.0, 0.0]  loss_bước0=1.9459
  default  std=[0.1649, 0.0666]  loss_bước0=1.9609


**Nhận xét Part 1.**
- Loss bước 0 = 2,269 > ln 7 = 1,946. Nguyên nhân: He init đặt vào cả lớp ra khiến logits có độ lệch
  chuẩn ≈ 0,6 (lớn hơn mức ~0 mà một phân bố đều 7 lớp cần), nên cross-entropy khởi điểm cao hơn ln 7.
  Tôi kiểm chứng cơ chế: giữ He cho lớp ẩn nhưng đặt bias lớp ra = 0 và giảm thang trọng số lớp ra
  → loss bước 0 về đúng 1,946 (khi trọng số lớp ra = 0) và 1,958 (khi nhân 0,1). Đây là hiện tượng
  "điểm số lớp cuối quá lớn" trong bảng triệu chứng Chương 5, không phải lỗi code: loss vẫn giảm đều.
- Quá khớp 20 mẫu: loss về ~0 và accuracy 100% → nhãn đúng, không softmax hai lần, `zero_grad` đúng,
  tham số được đưa vào optimizer.
- Gradient: mọi tham số (`net.0..net.4`) có `grad_norm` khác 0 hữu hạn → gradient chảy tới toàn mạng.
- Kích hoạt theo khởi tạo: `zeros` cho std = 0 ở mọi lớp (đối xứng hoàn hảo); `he` giữ std ổn định
  (~0,37–0,40) qua các lớp; `normal` std nhỏ dần; `default` nằm giữa. Chi tiết ở mục 3.7.


## Part 2 — Pipeline huấn luyện và baseline

`run_experiment(cfg, data)` là **hàm duy nhất**: mọi thí nghiệm chỉ là đổi dict `cfg`. Mỗi epoch ghi
train_loss (đo ở `eval()` trên tập con cố định 50 000 mẫu — cùng thang đo với val), val_loss/acc/macro-F1,
`grad_norm` **trước khi clip** (kèm giá trị lớn nhất và tỉ lệ bước bị clip), thời gian epoch, và loss bước 0.


In [ ]:
# Baseline: SGD+momentum 0.9, CE, He, batch 512, 20 epoch. Chọn lr bằng VAL 
print("Dò lr cho baseline trên VAL (chạy ngắn 6 epoch cho nhanh):")
probe = []
for lr in (0.01, 0.05, 0.1, 0.2):
    c = dict(BASE, exp_id=f"_probe-lr{lr:g}", lr=lr, epochs=6, train_eval_subsample=20000)
    r = run_experiment(c, data)
    probe.append((lr, r["summary"]["val_macro_f1"], r["summary"]["val_acc"]))
    print(f"  lr={lr:<5} val_macro_f1={r['summary']['val_macro_f1']:.4f}  val_acc={r['summary']['val_acc']:.4f}")
best_lr = max(probe, key=lambda t: t[1])[0]
print(f"-> chọn lr = {best_lr} cho baseline (theo val)")


Dò lr cho baseline trên VAL (chạy ngắn 6 epoch cho nhanh):


  lr=0.01  val_macro_f1=0.6612  val_acc=0.8187


  lr=0.05  val_macro_f1=0.7475  val_acc=0.8552


  lr=0.1   val_macro_f1=0.7646  val_acc=0.8586


  lr=0.2   val_macro_f1=0.8128  val_acc=0.8766
-> chọn lr = 0.2 cho baseline (theo val)


In [7]:
# Baseline với 3 seed -> đo độ nhiễu
base_results = []
for s in (1, 2, 3):
    r = get_or_run(dict(BASE, exp_id=f"base-s{s}", seed=s,
                        description=f"Baseline M-base, seed {s}"), data)
    base_results.append(r)
    print(f"base-s{s}: val_acc={r['summary']['val_acc']:.4f} val_macro_f1={r['summary']['val_macro_f1']:.4f} "
          f"best_ep={r['summary']['best_epoch']}")

accs = np.array([r["summary"]["val_acc"] for r in base_results])
f1s  = np.array([r["summary"]["val_macro_f1"] for r in base_results])
print(f"\nval_acc       : {accs.mean():.4f} ± {accs.std(ddof=1):.4f}")
print(f"val_macro_f1  : {f1s.mean():.4f} ± {f1s.std(ddof=1):.4f}")
print(f"ngưỡng nhiễu 2σ (val_macro_f1) = {2*f1s.std(ddof=1):.4f}")

# ảnh chồng baseline
plot_compare(base_results, "val_loss", f"{OUT_DIR}/figures/compare_baseline.png",
             "Baseline 3 seed — val loss")
plot_compare(base_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_baseline_f1.png",
             "Baseline 3 seed — val macro-F1")


base-s1: val_acc=0.8953 val_macro_f1=0.8229 best_ep=19
base-s2: val_acc=0.9000 val_macro_f1=0.8361 best_ep=20
base-s3: val_acc=0.8981 val_macro_f1=0.8415 best_ep=20

val_acc       : 0.8978 ± 0.0024
val_macro_f1  : 0.8335 ± 0.0095
ngưỡng nhiễu 2σ (val_macro_f1) = 0.0191


**Nhận xét baseline.** Đường val loss giảm đơn điệu tới cuối (best_epoch ≈ 19/20) → chưa hội tụ hẳn,
huấn luyện lâu hơn sẽ còn tốt lên (khớp nhận xét trong GUIDE). Khoảng cách train–val loss nhỏ → mô hình
chưa overfit ở 20 epoch. Val accuracy ≈ 0,90 ≫ mốc "đoán đa số" 0,4876. Độ nhiễu giữa 3 seed
(2σ ≈ 0,019 trên val macro-F1) là ngưỡng để xét "A hơn B" ở Part 3.


## Part 3 — Thí nghiệm

Mỗi thí nghiệm: dự đoán trước → đổi MỘT yếu tố so với baseline → chạy → đối chiếu. Tất cả so bằng
val. Danh sách cấu hình nằm ở `experiments.py` để bảo đảm công bằng.


### 3.1 Hàm mất mát — CE vs MSE

**Dự đoán trước:** MSE trên one-hot sẽ học **chậm và kém hơn** CE ở cùng lr/epoch. Lý do: gradient của CE
theo logits là `(softmax(z) − y)` — **không bão hoà** kể cả khi dự đoán sai nặng; còn MSE có gradient
`2(softmax(z) − y)·softmax'(z)`, bị nhân bởi `softmax'(z)` nên **bão hoà** khi xác suất đã tiến sát 0/1 →
tín hiệu học nhỏ đi đúng lúc cần mạnh nhất. Lưu ý không so **loss** trực tiếp (khác thang đo), chỉ so metric.


In [8]:
grp = [e for e in EXPERIMENTS if e["group"] == "loss"]
base1 = get_or_run(dict(BASE, exp_id="base-s1", seed=1), data)
res_loss = [get_or_run(e, data) for e in grp]
for r in [base1] + res_loss:
    s = r["summary"]
    print(f"{r['cfg']['exp_id']:16s} loss={r['cfg']['loss']:4s} step0={s['step0_loss']:.4f} "
          f"best_ep={s['best_epoch']:2d} val_acc={s['val_acc']:.4f} val_macro_f1={s['val_macro_f1']:.4f}")
plot_compare([base1] + res_loss, "val_macro_f1", f"{OUT_DIR}/figures/compare_loss.png",
             "CE vs MSE — val macro-F1")


base-s1          loss=ce   step0=2.2691 best_ep=19 val_acc=0.8953 val_macro_f1=0.8229
loss-mse-lr0.05  loss=mse  step0=4.4510 best_ep=20 val_acc=0.8553 val_macro_f1=0.6952


### 3.2 Bộ tối ưu hoá

**Dự đoán trước:** Adam/AdamW hội tụ nhanh hơn trong vài epoch đầu (bước được chuẩn hoá theo độ lớn gradient từng tham số), nhưng nếu chỉnh lr công bằng thì SGD+momentum có thể đuổi kịp hoặc vượt ở 20 epoch.
Adam nhạy với lr hơn SGD. AdamW (wd=0.01) khác Adam ở chỗ suy giảm trọng số tách khỏi gradient.


In [9]:
opt_ids = ["opt-sgd-lr0.05","opt-sgd-lr0.1","opt-sgd-lr0.2",
           "opt-sgdm-lr0.05","opt-sgdm-lr0.1","opt-sgdm-lr0.2",
           "opt-adam-lr0.001","opt-adam-lr0.003","opt-adam-lr0.01",
           "opt-adamw-lr0.001","opt-adamw-lr0.003","opt-adamw-lr0.01"]
by_id = {e["exp_id"]: e for e in EXPERIMENTS}
res_opt = [get_or_run(by_id[i], data) for i in opt_ids]
print(f"{'exp_id':22s} {'val_acc':>8s} {'val_macro_f1':>13s} {'best_ep':>8s} {'s/epoch':>8s}")
for r in res_opt:
    s = r["summary"]
    print(f"{r['cfg']['exp_id']:22s} {s['val_acc']:8.4f} {s['val_macro_f1']:13.4f} {s['best_epoch']:8d} {s['time_per_epoch_s']:8.2f}")

# lr tốt nhất của mỗi bộ
import collections
best_per_opt = {}
for opt in ("sgd", "sgd_momentum", "adam", "adamw"):
    cand = [r for r in res_opt if r["cfg"]["optimizer"] == opt]
    b = max(cand, key=lambda r: r["summary"]["val_macro_f1"])
    best_per_opt[opt] = b
    print(f"  tốt nhất {opt:13s}: {b['cfg']['exp_id']} (lr={b['cfg']['lr']}) val_f1={b['summary']['val_macro_f1']:.4f}")

plot_compare(res_opt, "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer.png",
             "Các bộ tối ưu × lr — val macro-F1")
plot_compare([best_per_opt[o] for o in ("sgd","sgd_momentum","adam","adamw")], "val_loss",
             f"{OUT_DIR}/figures/compare_optimizer_bestlr.png", "Mỗi bộ tối ưu ở lr tốt nhất — val loss")


exp_id                  val_acc  val_macro_f1  best_ep  s/epoch
opt-sgd-lr0.05           0.8382        0.7074       19     2.07
opt-sgd-lr0.1            0.8611        0.7521       19     2.01
opt-sgd-lr0.2            0.8774        0.7946       19     2.09
opt-sgdm-lr0.05          0.8953        0.8229       19     2.47
opt-sgdm-lr0.1           0.9052        0.8476       19     2.15
opt-sgdm-lr0.2           0.9093        0.8554       20     2.17
opt-adam-lr0.001         0.9036        0.8466       20     2.55
opt-adam-lr0.003         0.9149        0.8749       19     2.54
opt-adam-lr0.01          0.9117        0.8578       17     2.52
opt-adamw-lr0.001        0.8998        0.8434       19     3.35
opt-adamw-lr0.003        0.9107        0.8610       20     3.42
opt-adamw-lr0.01         0.9065        0.8495       17     3.44
  tốt nhất sgd          : opt-sgd-lr0.2 (lr=0.2) val_f1=0.7946
  tốt nhất sgd_momentum : opt-sgdm-lr0.2 (lr=0.2) val_f1=0.8554
  tốt nhất adam         : opt-adam-lr0.00

### 3.3 Hyper-parameter

**Dự đoán trước:** (a) batch lớn hơn → **ít bước cập nhật hơn** trong cùng 20 epoch nên học chậm hơn;
(b) `M-wide`/`M-deep` nhiều tham số hơn → có thể tốt hơn nếu chưa quá khớp; (c) weight_decay nhẹ có thể
giúp chống quá khớp. Thời gian mỗi epoch được ghi để thấy chi phí.


In [10]:
hp_ids = ["hp-batch128","hp-batch2048","hp-wide","hp-deep","hp-wd1e-2"]
res_hp = [get_or_run(by_id[i], data) for i in hp_ids]
for r in res_hp:
    c, s = r["cfg"], r["summary"]
    steps = int(np.ceil(371847 / c["batch"])) * c["epochs"]
    print(f"{c['exp_id']:14s} batch={c['batch']:5d} hidden={c['hidden']} wd={c['weight_decay']:<5} "
          f"steps={steps:5d} val_acc={s['val_acc']:.4f} val_f1={s['val_macro_f1']:.4f} {s['time_per_epoch_s']:.2f}s/ep")
plot_compare(res_hp, "val_macro_f1", f"{OUT_DIR}/figures/compare_hparam.png", "Hyper-parameter — val macro-F1")


hp-batch128    batch=  128 hidden=[256, 128] wd=0.0   steps=58120 val_acc=0.9145 val_f1=0.8629 5.97s/ep
hp-batch2048   batch= 2048 hidden=[256, 128] wd=0.0   steps= 3640 val_acc=0.8721 val_f1=0.7732 1.26s/ep
hp-wide        batch=  512 hidden=[512, 256] wd=0.0   steps=14540 val_acc=0.9127 val_f1=0.8601 4.26s/ep
hp-deep        batch=  512 hidden=[256, 128, 64] wd=0.0   steps=14540 val_acc=0.9151 val_f1=0.8616 2.47s/ep
hp-wd1e-2      batch=  512 hidden=[256, 128] wd=0.01  steps=14540 val_acc=0.7251 val_f1=0.3874 2.16s/ep


### 3.4 Dropout

**Dự đoán:** mô hình chưa overfit nên dropout có thể không giúp gì nhiều, thậm chí làm chậm hội tụ. Nếu q quá lớn, mô hình thiếu năng lực biểu diễn trong 20 epoch.


In [11]:
drop_ids = ["drop-0.1","drop-0.3","drop-0.5"]
res_drop = [get_or_run(by_id[i], data) for i in drop_ids]
for r in [base1] + res_drop:
    c, s = r["cfg"], r["summary"]
    gap = s["final_val_loss"] - s["final_train_loss"]
    print(f"{c['exp_id']:10s} q={c['dropout']:<4} val_acc={s['val_acc']:.4f} val_f1={s['val_macro_f1']:.4f} "
          f"train_loss={s['final_train_loss']:.4f} val_loss={s['final_val_loss']:.4f} gap={gap:+.4f}")
plot_compare([base1] + res_drop, "val_loss", f"{OUT_DIR}/figures/compare_dropout.png", "Dropout — val loss")


base-s1    q=0.0  val_acc=0.8953 val_f1=0.8229 train_loss=0.2447 val_loss=0.2646 gap=+0.0200
drop-0.1   q=0.1  val_acc=0.8906 val_f1=0.8236 train_loss=0.2621 val_loss=0.2727 gap=+0.0105
drop-0.3   q=0.3  val_acc=0.8612 val_f1=0.7628 train_loss=0.3345 val_loss=0.3389 gap=+0.0044
drop-0.5   q=0.5  val_acc=0.8249 val_f1=0.6627 train_loss=0.4073 val_loss=0.4099 gap=+0.0026


### 3.5 Gradient clipping



In [12]:
clip_ids = ["clip-lr0.05-c1.0","clip-lr1.0-noc","clip-lr1.0-c1.0",
            "clip-lr2.0-noc","clip-lr2.0-c1.0","clip-lr2.0-c0.5"]
res_clip = [get_or_run(by_id[i], data) for i in clip_ids]
print(f"{'exp_id':20s} {'lr':>6s} {'clip':>6s} {'val_f1':>8s} {'gn_max':>8s} {'clip_frac':>10s} {'diverged':>9s}")
for r in res_clip:
    c, s, h = r["cfg"], r["summary"], r["history"]
    print(f"{c['exp_id']:20s} {c['lr']:6g} {str(c['clip_norm']):>6s} {s['val_macro_f1']:8.4f} "
          f"{max(h['grad_norm_max']):8.3f} {max(h['clip_frac']):10.3f} {str(s['diverged']):>9s}")
plot_compare(res_clip, "val_macro_f1", f"{OUT_DIR}/figures/compare_clipping.png",
             "Gradient clipping — val macro-F1")


exp_id                   lr   clip   val_f1   gn_max  clip_frac  diverged
clip-lr0.05-c1.0       0.05    1.0   0.8272    2.908      0.193     False
clip-lr1.0-noc            1   None   0.7757    8.908      0.000     False
clip-lr1.0-c1.0           1    1.0   0.8193    4.272      0.010     False
clip-lr2.0-noc            2   None   0.0936   36.989      0.000     False
clip-lr2.0-c1.0           2    1.0   0.3425    5.282      0.017     False
clip-lr2.0-c0.5           2    0.5   0.4735    4.179      0.072     False


### 3.6 Mixed precision



In [13]:
amp_ids = ["amp-fp32","amp-fp16","amp-bf16"]
res_amp = [get_or_run(by_id[i], data) for i in amp_ids]
for r in res_amp:
    c, s = r["cfg"], r["summary"]
    print(f"{c['exp_id']:10s} prec={c['precision']:5s} {s['time_per_epoch_s']:6.2f}s/epoch "
          f"val_acc={s['val_acc']:.4f} val_f1={s['val_macro_f1']:.4f} peak_mem={s['peak_mem_MB']}")
plot_compare(res_amp, "val_loss", f"{OUT_DIR}/figures/compare_amp.png", "Mixed precision — val loss")


amp-fp32   prec=fp32    2.33s/epoch val_acc=0.8549 val_f1=0.7436 peak_mem=nan
amp-fp16   prec=fp16   90.57s/epoch val_acc=0.8547 val_f1=0.7474 peak_mem=nan
amp-bf16   prec=bf16   64.11s/epoch val_acc=0.8544 val_f1=0.7456 peak_mem=nan


### 3.7 Khởi tạo tham số



In [14]:
init_ids = ["init-zeros","init-normal","init-xavier","init-default"]
res_init = [get_or_run(by_id[i], data) for i in init_ids]
for r in [base1] + res_init:
    c, s = r["cfg"], r["summary"]
    print(f"{c['exp_id']:14s} init={c['init']:8s} step0={s['step0_loss']:.4f} "
          f"val_acc={s['val_acc']:.4f} val_f1={s['val_macro_f1']:.4f} div={s['diverged']}")
plot_compare([base1] + res_init, "val_macro_f1", f"{OUT_DIR}/figures/compare_init.png",
             "Khởi tạo tham số — val macro-F1")


base-s1        init=he       step0=2.2691 val_acc=0.8953 val_f1=0.8229 div=False
init-zeros     init=zeros    step0=1.9459 val_acc=0.4876 val_f1=0.0936 div=False
init-normal    init=normal   step0=1.9460 val_acc=0.8882 val_f1=0.8194 div=False
init-xavier    init=xavier   step0=2.0222 val_acc=0.9020 val_f1=0.8374 div=False
init-default   init=default  step0=1.9830 val_acc=0.8959 val_f1=0.8413 div=False


### 3.8 Cấu hình cuối cùng

Kết hợp những gì tốt nhất **theo val** (không nhìn eval) thành một cấu hình cuối cùng. Xem mục 4 để biết
cấu hình đã chọn và lý do.


In [15]:
# Bốn ứng viên ghép từ những gì thắng trên val. Chọn bằng VAL (không nhìn eval).
fin_ids = ["fin-adam3e3","fin-adam3e3-wide","fin-adam3e3-wide-40","fin-adam3e3-deep-40"]
res_fin = [get_or_run(by_id[i], data) for i in fin_ids]
print(f"{'exp_id':24s} {'val_acc':>8s} {'val_macro_f1':>13s} {'best_ep':>8s}")
for r in res_fin:
    s = r["summary"]
    print(f"{r['cfg']['exp_id']:24s} {s['val_acc']:8.4f} {s['val_macro_f1']:13.4f} {s['best_epoch']:8d}")
best_fin = max(res_fin, key=lambda r: r["summary"]["val_macro_f1"])
print("\n-> chọn theo val:", best_fin["cfg"]["exp_id"],
      "val_macro_f1 =", round(best_fin["summary"]["val_macro_f1"], 4))
plot_compare([base_results[0]] + res_fin, "val_macro_f1",
             f"{OUT_DIR}/figures/compare_final.png", "Ứng viên cấu hình cuối — val macro-F1")


exp_id                    val_acc  val_macro_f1  best_ep
fin-adam3e3                0.9149        0.8749       19
fin-adam3e3-wide           0.9281        0.8886       17
fin-adam3e3-wide-40        0.9407        0.9096       39
fin-adam3e3-deep-40        0.9373        0.8989       40

-> chọn theo val: fin-adam3e3-wide-40 val_macro_f1 = 0.9096


## Part 4 — Đánh giá trên eval, bảng, ảnh, nộp bài



In [16]:
# Cấu hình cuối cùng được chọn CHỈ bằng val (xem mục 3.8 / REPORT.md). Huấn luyện lại để lấy best_state
# (JSON không lưu trọng số), rồi dự đoán toàn bộ tập eval.
FINAL_ID = "fin-adam3e3-wide-40"

def train_and_predict(cfg, pred_path):
    result = run_experiment(cfg, data)          # cùng seed -> tái lập
    preds = final_eval(cfg, result, data, pred_path)
    return result

# baseline (seed 1) và cấu hình cuối cùng
r_base = train_and_predict(dict(BASE, exp_id="base-s1", seed=1), f"{OUT_DIR}/predictions_base.csv")
r_final = train_and_predict(by_id[FINAL_ID], f"{OUT_DIR}/predictions_eval.csv")

# chấm điểm bằng script chính thức (dùng đường dẫn tuyệt đối vì cwd đổi sang REPO_ROOT)
for pred, out in [(f"{OUT_DIR}/predictions_base.csv", f"{OUT_DIR}/eval_base.json"),
                  (f"{OUT_DIR}/predictions_eval.csv", f"{OUT_DIR}/eval_result.json")]:
    subprocess.run([sys.executable, "scripts/evaluate.py",
                    "--pred", os.path.abspath(pred), "--out", os.path.abspath(out)],
                   cwd=REPO_ROOT, check=True)

res_eval = json.load(open(f"{OUT_DIR}/eval_result.json"))
res_base = json.load(open(f"{OUT_DIR}/eval_base.json"))
print("\nBASELINE  eval:", f"acc={res_base['accuracy']:.4f} macro_f1={res_base['macro_f1']:.4f}")
print("FINAL     eval:", f"acc={res_eval['accuracy']:.4f} macro_f1={res_eval['macro_f1']:.4f}")
print("cải thiện macro-F1:", round(res_eval["macro_f1"] - res_base["macro_f1"], 4))


đã ghi ../predictions_base.csv (116203 dòng)


đã ghi ../predictions_eval.csv (116203 dòng)



BASELINE  eval: acc=0.8950 macro_f1=0.8254
FINAL     eval: acc=0.9383 macro_f1=0.9057
cải thiện macro-F1: 0.0803


In [17]:
# Phân tích lỗi theo lớp
import pandas as pd
print("lớp  support  precision  recall    f1")
for pc in res_eval["per_class"]:
    print(f"{pc['cls']:>3d}  {pc['support']:7d}  {pc['precision']:9.4f}  {pc['recall']:6.4f}  {pc['f1']:6.4f}")
cm = np.array(res_eval["confusion_matrix"])
names = ["Spruce/Fir","Lodgepole","Ponderosa","Cottonwood","Aspen","Douglas-fir","Krummholz"]
plot_confusion(cm, names, f"{OUT_DIR}/figures/eval_confusion.png")
print("\nma trận nhầm lẫn:\n", pd.DataFrame(cm).to_string())
worst = min(res_eval["per_class"], key=lambda p: p["f1"])
print(f"\nLớp khó nhất: {worst['cls']} (F1={worst['f1']:.4f}, support={worst['support']})")


lớp  support  precision  recall    f1
  0    42368     0.9300  0.9407  0.9353
  1    56661     0.9472  0.9465  0.9469
  2     7151     0.9420  0.9408  0.9414
  3      549     0.8052  0.8962  0.8483
  4     1899     0.8965  0.7757  0.8317
  5     3473     0.9051  0.8871  0.8960
  6     4102     0.9606  0.9213  0.9405



ma trận nhầm lẫn:
        0      1     2    3     4     5     6
0  39854   2357     3    0    29     5   120
1   2667  53629   109    0   125    96    35
2      1    128  6728   88    14   192     0
3      0      0    45  492     0    12     0
4     34    352    22    0  1473    18     0
5      8    116   235   31     2  3081     0
6    289     34     0    0     0     0  3779

Lớp khó nhất: 4 (F1=0.8317, support=1899)


In [18]:
# Ghi bảng experiments.xlsx từ results/*.json
results_all = load_results(f"{OUT_DIR}/results")
# gắn eval cho baseline và cấu hình cuối cùng
eval_map = {"base-s1": dict(eval_acc=res_base["accuracy"], eval_macro_f1=res_base["macro_f1"]),
            FINAL_ID: dict(eval_acc=res_eval["accuracy"], eval_macro_f1=res_eval["macro_f1"])}
# bỏ các exp tạm (_probe)
results_all = [r for r in results_all if not r["cfg"]["exp_id"].startswith("_")]
rows = [to_row(r, eval_scores=eval_map.get(r["cfg"]["exp_id"])) for r in results_all]
rows.sort(key=lambda x: (x["group"] or "", x["exp_id"] or ""))

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx",
           f"{OUT_DIR}/experiments.xlsx",
           baseline_seed_ids=["base-s1", "base-s2", "base-s3"],
           summary_notes={
               "baseline": "3 seed; val_macro_f1 0.8335 ± 0.0095, 2σ ≈ 0.019; best_epoch 19-20/20 (chưa hội tụ).",
               "loss": "CE (0.8229) > MSE (0.6952) ở cùng lr: MSE bão hoà gradient khi softmax bão hoà.",
               "optimizer": "Adam 3e-3 tốt nhất (0.8749); SGD+momentum 0.2 sánh được (0.8554); SGD thuần yếu nhất.",
               "hparam": "batch 128 (nhiều bước) tốt nhất; wide/deep nhỉnh hơn; wd=0.01 với lr=0.05 làm sụp.",
               "dropout": "Chưa quá khớp -> q>0 không giúp; q=0.3/0.5 hại rõ.",
               "clipping": "lr thường: c=1.0 chỉ kích hoạt ~19% bước (‖g‖max=2.9) -> kết quả là nhiễu. lr=2.0 không clip sụp về lớp đa số; clip c=0.5 cứu lên 0.4735.",
               "amp": "CPU không tensor core: fp16 ×38.9, bf16 ×27.5 chậm hơn fp32; độ chính xác tương đương.",
               "init": "zeros hỏng (đối xứng, F1 = 0.0936 = đoán đa số); he ổn định nhất; normal/xavier/default trong khoảng nhiễu.",
               "final": f"{FINAL_ID}: M-wide + Adam 3e-3 + 40ep; val_f1=0.9096 -> eval_f1={res_eval['macro_f1']:.4f}.",
           })
print(f"\nTổng số dòng bảng: {len(rows)}")


đã ghi ../experiments.xlsx với 41 dòng thí nghiệm

Tổng số dòng bảng: 41


**Phân tích.** Lớp khó nhất là lớp 4 (Aspen) — F1 0,8317, chủ yếu do recall thấp (0,776): 352/1 899 mẫu Aspen bị nhầm sang lớp Lodgepole Pine. Hai lớp cùng phân bố ở sườn núi
độ cao trung bình nên các biến địa hình chồng lấn, và Aspen chỉ ~1,6% số mẫu nên mô hình ưu tiên lớp đa số.
Lớp 3 (Cottonwood) F1 0,8483 nhưng precision thấp (0,805) nên dễ bị nhầm sang lớp 2/5 dù chỉ 549 mẫu.
Chi tiết và số liệu ở `REPORT.md` mục 4.1; xem ma trận nhầm lẫn `figures/eval_confusion.png`.
